# 15 — Evaluation and bootstrap sensitivity

This post-protocol audit uses saved development-panel forecasts; it does not replace the frozen confirmatory outcomes. It reports MSE and RMSE with explicit definitions, adds a dependence-sensitive variogram score, and compares hierarchical-bootstrap conclusions over block lengths 10, 20, and 40. Test NLL and analytic Gaussian projected PITs are generated by the current fitting code when notebooks 08 and 09 are rerun; saved forecast samples alone cannot recover an exact Gaussian predictive density or analytic PIT.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from innovcal.evaluation import paired_seed_block_bootstrap, evaluate_samples

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'results' / 'evaluation_sensitivity'
OUT.mkdir(parents=True, exist_ok=True)
ARCHIVE = np.load(ROOT / 'results' / 'seed_uncertainty' / 'seed_forecasts.npz')
MANIFEST = pd.read_csv(ROOT / 'results' / 'seed_uncertainty' / 'seed_forecast_manifest.csv')
full_label = 'CA-RNN sequential' if 'CA-RNN sequential' in set(MANIFEST.model) else 'CA-RNN'
keys = dict(zip(MANIFEST.model, MANIFEST.array_key))
target = ARCHIVE['target']
projections = ARCHIVE['projections']
forecasts = {'RNN': ARCHIVE[keys['RNN']], 'CA-RNN': ARCHIVE[keys[full_label]]}
assert forecasts['RNN'].shape == forecasts['CA-RNN'].shape
print('Matched seeds, draws, origins, dimensions:', forecasts['RNN'].shape)

Matched seeds, draws, origins, dimensions: (10, 500, 957, 4)


In [6]:
rows = []
for model, all_seeds in forecasts.items():
    for seed_index, samples in enumerate(all_seeds):
        values = evaluate_samples(target, samples, projections=projections)
        assert np.isclose(values['rmse'] ** 2, values['mean_squared_error'])
        rows.append({'model': model, 'seed_index': seed_index, **values})
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT / 'seed_metric_audit.csv', index=False)
display(metrics.groupby('model')[['mean_squared_error', 'rmse', 'energy_score', 'variogram_score']].mean())
print('MSE is mean squared component error; RMSE is its square root computed within each seed.')

,mean_squared_error,rmse,energy_score,variogram_score
model,,,,
CA-RNN,0.898605,0.947946,1.153081,0.156705
RNN,0.883330,0.939856,1.145041,0.156821


MSE is mean squared component error; RMSE is its square root computed within each seed.


In [7]:
saved_metrics = pd.read_csv(ROOT / 'results' / 'seed_uncertainty' / 'seed_metrics.csv')
new_columns = ['test_nll', 'analytic_pit_calibration_error', 'analytic_pit_mean_absolute_autocorrelation']
if set(new_columns).issubset(saved_metrics.columns):
    full_model = 'CA-RNN sequential' if 'CA-RNN sequential' in set(saved_metrics.model) else 'CA-RNN'
    analytic = saved_metrics.loc[saved_metrics.model.isin(['RNN', full_model]), ['model', 'training_seed', *new_columns]].copy()
    analytic.loc[analytic.model.eq(full_model), 'model'] = 'CA-RNN'
    analytic.to_csv(OUT / 'analytic_pit_and_test_nll.csv', index=False)
    display(analytic.groupby('model')[new_columns].mean())
else:
    print('Saved fits predate analytic PIT and test NLL fields. Rerun notebook 08 to populate them; sample draws cannot recover exact Gaussian parameters.')

,test_nll,analytic_pit_calibration_error,analytic_pit_mean_absolute_autocorrelation
model,,,
CA-RNN,5.309488,0.001223,0.036909
RNN,5.271313,0.001439,0.030724


In [8]:
frames = []
for block_length in (10, 20, 40):
    result = paired_seed_block_bootstrap(
        target, forecasts, [('CA-RNN', 'RNN')], projections,
        block_length=block_length, n_bootstrap=1000, confidence=0.95, seed=1500 + block_length,
    )
    frames.append(result)
sensitivity = pd.concat(frames, ignore_index=True)
sensitivity['excludes_zero'] = (sensitivity.ci_lower > 0) | (sensitivity.ci_upper < 0)
sensitivity.to_csv(OUT / 'block_length_sensitivity.csv', index=False)
display(sensitivity.loc[sensitivity.metric.isin(['mean_squared_error', 'energy_score', 'variogram_score', 'interval_score', 'pit_calibration_error'])])

,n_seeds,n_origins,block_length,n_bootstrap,candidate,comparator,metric,difference,bootstrap_se,ci_lower,ci_upper,probability_difference_below_zero,excludes_zero
0,10,957,10,1000,CA-RNN,RNN,mean_squared_error,0.015275,0.001493,0.012540,0.018345,0.000,True
1,10,957,10,1000,CA-RNN,RNN,energy_score,0.008040,0.000856,0.006480,0.009881,0.000,True
2,10,957,10,1000,CA-RNN,RNN,variogram_score,-0.000116,0.000227,-0.000409,0.000447,0.464,False
3,10,957,10,1000,CA-RNN,RNN,interval_score,-0.003866,0.008560,-0.019534,0.013328,0.598,False
7,10,957,10,1000,CA-RNN,RNN,pit_calibration_error,-0.000213,0.000041,-0.000319,-0.000157,1.000,True
9,10,957,20,1000,CA-RNN,RNN,mean_squared_error,0.015275,0.001526,0.012300,0.018164,0.000,True
10,10,957,20,1000,CA-RNN,RNN,energy_score,0.008040,0.000827,0.006503,0.009783,0.000,True
11,10,957,20,1000,CA-RNN,RNN,variogram_score,-0.000116,0.000227,-0.000383,0.000488,0.366,False
12,10,957,20,1000,CA-RNN,RNN,interval_score,-0.003866,0.008076,-0.018150,0.012624,0.606,False
16,10,957,20,1000,CA-RNN,RNN,pit_calibration_error,-0.000213,0.000040,-0.000326,-0.000171,1.000,True


The primary confirmatory bootstrap remains the frozen 20-day, 2,000-draw protocol. If signs or zero-exclusion decisions change with block length here, report the development-panel inference as sensitive. Analytic PIT and NLL values require retained Gaussian parameters or refitting, so neither is approximated from the 500 saved forecast draws in this audit.